# Stuttering Detection: Linear Classification Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Logistic Regression vs. The Perceptron

---

## Step 1: Environment Setup

In [1]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import LogisticModel, PerceptronModel

# Experiment Paths
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 (Optional): Operational Mode for Data Extraction
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Loading and Preparation

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading individual .npy files into training matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Oversampling to handle imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standardization
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Logistic Regression

In [3]:
log_model = LogisticModel("Logistic_Regression")
log_model.train(X_train_final, y_train_bal)
log_model.evaluate(X_test_final, y_test)

[Model: Logistic_Regression] Initialized.
[Logistic_Regression] Training on 4440 samples...

--- Evaluation: Logistic_Regression ---
Accuracy: 0.6640
Precision: 0.5231
Recall: 0.6830
F1: 0.5925

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      311             165            
True: Stutter(1)     84              181            


{'accuracy': 0.6639676113360324,
 'precision': 0.523121387283237,
 'recall': 0.6830188679245283,
 'f1': 0.5924713584288053,
 'confusion_matrix': array([[311, 165],
        [ 84, 181]])}

## Step 5: Model 2 - The Perceptron

In [4]:
perc_model = PerceptronModel("Iterative_Perceptron")
perc_model.train(X_train_final, y_train_bal)
perc_model.evaluate(X_test_final, y_test)

[Model: Iterative_Perceptron] Initialized.
[Iterative_Perceptron] Training on 4440 samples...

--- Evaluation: Iterative_Perceptron ---
Accuracy: 0.6235
Precision: 0.4816
Recall: 0.6906
F1: 0.5674

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      279             197            
True: Stutter(1)     82              183            


{'accuracy': 0.6234817813765182,
 'precision': 0.48157894736842105,
 'recall': 0.690566037735849,
 'f1': 0.5674418604651162,
 'confusion_matrix': array([[279, 197],
        [ 82, 183]])}